In [1]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from itertools import combinations
from itertools import product

from sklearn.utils.parallel import Parallel, delayed

%run Model_functions.ipynb

In [2]:
BASE_DIR = "../../DATA/AGB_DATA/Merged_Data/"
#BASE_DIR = "../../Data/"

# SENTINEL + CANOPY DATA

In [4]:
SENTINEL_DATA_CSV        = BASE_DIR + "/Sentinel_AGB/AGB_SENTINEL_CANOPY.csv"
sentinel_raw_df = pd.read_csv(SENTINEL_DATA_CSV)
print(sentinel_raw_df.shape)
assert len(sentinel_raw_df["simard_height_m"].head())
assert len(sentinel_raw_df["tandemx_height_m"].head())

(8774, 34)


In [5]:
sentinel_raw_df.columns

Index(['dataset', 'plot_id', 'start_date', 'end_date', 'latitude', 'longitude',
       'diameter', 'height', 'species', 'plant_AGB_kg', 'capture_start',
       'capture_end', 'sentinel_time', 'Blue', 'Green', 'Red', 'RE1', 'RE2',
       'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1',
       'NDRE2', 'NDRE3', 'CIrededge', 'CLOUD_COVERAGE', 'cloud_threshold_used',
       'simard_height_m', 'tandemx_height_m'],
      dtype='object')

## Sentinel feature selection

In [7]:
sent_non_feature_cols = [
    'plant_AGB_kg',        # Target variable
    'dataset',             # metadata
    'sentinel_time',       # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'diameter',            # Allometric
    'height',               # Allometric
    'cloud_threshold_used',
    'start_date'
]

SENT_interact_cols = [
    # Indices (excluding MNDWI which isn't a biomass index)
    'EVI', 'NBR', 'NDVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge',
    # Raw structural bands
    'NIR', 'SWIR1', 'SWIR2',
]

target = 'plant_AGB_kg'

sent_feature_cols = [c for c in sentinel_raw_df.columns if c not in sent_non_feature_cols]

### Feature selection with Simard canopy height

In [9]:
X_sent_t = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][sent_feature_cols].copy()

# Select TANDEMX
X_sent_t = X_sent_t.rename({'tandemx_height_m': 'height'}, axis=1)
X_sent_t = X_sent_t.drop(columns=['simard_height_m'])

In [10]:
X_sent_t.shape

(3880, 21)

In [11]:
X_sent_t.columns

Index(['plot_id', 'species', 'Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3',
       'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1',
       'NDRE2', 'NDRE3', 'CIrededge', 'CLOUD_COVERAGE', 'height'],
      dtype='object')

### Create interaction terms

In [13]:
X_sent_t = create_interact_terms(X_sent_t, SENT_interact_cols)

### Feature selection with Simard canopy height

In [15]:
X_sent_s = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][sent_feature_cols].copy()

# Select SIMARD
X_sent_s = X_sent_s.rename({'simard_height_m': 'height'}, axis=1)
X_sent_s = X_sent_s.drop(columns=['tandemx_height_m'])

### Create interaction terms

In [17]:
X_sent_s = create_interact_terms(X_sent_s, SENT_interact_cols)

### Create target variable

In [19]:
y_sent = sentinel_raw_df[sentinel_raw_df['dataset'] == 'Belige'][target].copy()

## Sentinel test features

In [21]:
test_cv = 5

In [22]:
sent_struct_features = ['height']
species              = ['species']

sent_bands   = ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2']
sent_indices = ['NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']

sent_top_spectral  = ['EVI', 'NIR', 'NBR', 'SWIR1']
sent_redband       = ['RE1', 'RE2', 'RE3']
sent_interaction_terms = [c for c in X_sent_t.columns if c.startswith('height_x_')]

# Curated interactions matching curated spectral sets
sent_top_spectral_interactions = [f'height_x_{b}' for b in sent_top_spectral
                                  if f'height_x_{b}' in sentinel_raw_df.columns]
sent_redband_interactions      = [f'height_x_{b}' for b in sent_redband
                                  if f'height_x_{b}' in sentinel_raw_df.columns]

sent_features_list = [
    # Non-spectral baselines
    #struct_features,  # already tested in EMIT
    #species,          # already tested in EMIT
    sent_struct_features + species,

    # Spectral alone
    sent_bands,
    sent_indices,
    sent_top_spectral,
    sent_redband,

    # Spectral + height
    sent_bands        + sent_struct_features,
    sent_indices      + sent_struct_features,
    sent_top_spectral + sent_struct_features,
    sent_redband      + sent_struct_features,

    # Spectral + height + species
    sent_bands        + sent_struct_features + species,
    sent_top_spectral + sent_struct_features + species,

    # Spectral + height + interactions (curated where it matters)
    sent_bands        + sent_struct_features + sent_interaction_terms,
    sent_top_spectral + sent_struct_features + sent_top_spectral_interactions,
    sent_redband      + sent_struct_features + sent_redband_interactions,

    # Full specturm of features
    sent_bands        + sent_indices + sent_struct_features + sent_interaction_terms + species
]

## Sentiel. Create plot groups.

In [24]:
# Retain the groups/plot_id for splitting the data based on groups.
plot_groups_sent = None
if 'plot_id' in X_sent_t.columns:
    plot_groups_sent = X_sent_t['plot_id'].copy()
    X_sent_t = X_sent_t.drop(columns=['plot_id'])

if 'plot_id' in X_sent_s.columns:
    X_sent_s = X_sent_s.drop(columns=['plot_id'])

near_zero_sites_sent, high_agb_sites_sent,\
    near_zero_plots_sent, high_agb_plots_sent = \
        get_low_and_high_agb_plots(y_sent,
                                   plot_groups_sent)

grp_sentinel = GROUP_INFO(near_zero_sites_sent,
                          high_agb_sites_sent,
                          near_zero_plots_sent,
                          high_agb_plots_sent,
                          groups=plot_groups_sent,
                          cv=test_cv)

High-AGB threshold  : 104.49 kg
Near-zero threshold : 0.004134

Near-zero variance plots:
  Big Creek_1               : log var = 0.000036
  Big Creek_4               : log var = 0.000032
  Frenchman Caye_1          : log var = 0.000753
  Frenchman Caye_2          : log var = 0.000381
  Frenchman Caye_3          : log var = 0.000693
  Frenchman Caye_4          : log var = 0.001306
  Frenchman Caye_5          : log var = 0.001283
  Frenchman Caye_6          : log var = 0.000158
  Shipstern Lagoon_1        : log var = 0.001064
  Shipstern Lagoon_3        : log var = 0.000232
  Shipstern Lagoon_4        : log var = 0.000113
  Shipstern Lagoon_5        : log var = 0.000052
  Shipstern Lagoon_6        : log var = 0.000135

High-AGB plots:
  Channel Caye_1            : max AGB = 310.9 kg
  Channel Caye_2            : max AGB = 206.4 kg
  Channel Caye_3            : max AGB = 427.2 kg
  Channel Caye_4            : max AGB = 237.6 kg
  Channel Caye_5            : max AGB = 170.4 kg
  Channel C

In [25]:
assert plot_groups_sent is not None

# SENTINEL-2 EXPERIMENTS

In [27]:
test_cv = 5

In [28]:
LINEAR_FULL    = ["linear", "ridge", "lasso", "elasticnet"]
LINEAR_NO_OLS  = ["ridge", "lasso", "elasticnet"]

data_combos= [("SENTINEL + TANDEMX", X_sent_t, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL),
              ("SENTINEL + SIMARD" , X_sent_s, y_sent, sent_features_list, grp_sentinel, LINEAR_FULL)
             ]
model_list = ["linear", "rf", "xgboost", "lgbm", "merf"]
is_grids   = [False, True]
is_groups  = [True]

In [29]:
global_experiment_list = {}
experiments_by_category = {}

for (combo_name, X, y, features_list, grp, linear_variants), model, use_grid, use_groups in product(
    data_combos, model_list, is_grids, is_groups
):
    # Linear models do not benefit from grid search.
    if model == 'linear' and use_grid:
        continue
    
    print(f"\n{'='*70}")
    print(f"DATA: {combo_name} | MODEL: {model} | GRID: {use_grid} | GROUPED: {use_groups}")
    print('='*70)

    #run_label = f"{combo_name}, {model}, grid_{'yes' if use_grid else 'no'}, GROUPED: {'yes' if use_groups else 'no'}"

    exp_result = run_experiment(
        X, y,
        is_groups       = use_groups,
        group_info      = grp,
        features_list   = features_list,
        model_type      = model,
        linear_variants = linear_variants,
        is_grid         = use_grid,
        is_stratified   = False,
        exp_total_list = global_experiment_list,
        exp_by_category=experiments_by_category,
        dataset        = combo_name
    )


DATA: SENTINEL + TANDEMX | MODEL: linear | GRID: False | GROUPED: True

[1/60]

 EXPERIMENT-1. SENTINEL + TANDEMX,Model: LINEAR REGRESSION, Grouping? Yes, Hypertuned? No, Features: ['height', 'species']
Test R²     : 0.0724
Test RMSE   : 4.45 kg
Train R² (log scale): 0.2886
Train R² (orig scale): -0.0202
Train RMSE  : 18.76 kg
Num rows    : 3100
Num Features: 4

 Cross-validation ---
CV R² mean: 0.2144
CV R² std : 0.2030
CV scores : [-0.023  0.59   0.182  0.133  0.189]

Grouped Cross-validation ---
Grouped CV R² mean: 0.2626
Grouped CV R² std : 0.2058
Grouped CV scores : [-0.046  0.186  0.123 -0.01   0.394  0.54   0.621  0.221  0.268  0.328]
 EVALUATION: EXPERIMENT-1. SENTINEL + TANDEMX,Model: LINEAR REGRESSION, Grouping? Yes, Hypertuned? No, Features: ['height', 'species']

Test set:
  R²   : 0.072
  RMSE : 4.45 kg

 ✅ Test R² is positive (0.072)

Regular CrossValidation:
  Mean   : 0.214
  Std    : 0.203
  Scores : [-0.023  0.59   0.182  0.133  0.189]
 ✅ CV mean is positive (0.214)


C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'subsample': 0.6, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}

 XGBOOST: EXPERIMENT-1. SENTINEL + TANDEMX,Model: XGBOOST, Grouping? Yes, Hypertuned? Yes, Features: ['height', 'species']
Test R²     : 0.0553
Test RMSE   : 4.49 kg
Train R² (log scale): 0.4186
Train R² (orig scale): 0.1310
Train RMSE  : 17.31 kg
Num rows    : 3100
Num Features: 4

 Cross-validation ---
CV R² mean: 0.1922
CV R² std : 0.0939
CV scores : [0.041 0.319 0.245 0.147 0.208]

Grouped Cross-validation ---
Grouped CV R² mean: 0.2918
Grouped CV R² std : 0.1533
Grouped CV scores : [ 0.163  0.247  0.297 -0.058  0.441  0.535  0.366  0.326  0.246  0.356]
 EVALUATION: EXPERIMENT-1. SENTINEL + TANDEMX,Model: XGBOOST, Grouping? Yes, Hypertuned? Yes, Features: ['height', 'species']

Test set:
  R²   : 0.055
  RMSE : 4.49 kg

 ✅ Test R² is positive (0.055)

Regular CrossValidation:
  Mean   : 0.192
  Std    : 0.09

C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'subsample': 0.6, 'reg_lambda': 5, 'reg_alpha': 1.0, 'n_estimators': 100, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.01, 'colsample_bytree': 1.0}

 XGBOOST: EXPERIMENT-3. SENTINEL + TANDEMX,Model: XGBOOST, Grouping? Yes, Hypertuned? Yes, Features: ['NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']
Test R²     : 0.1313
Test RMSE   : 4.30 kg
Train R² (log scale): 0.3299
Train R² (orig scale): 0.0194
Train RMSE  : 18.39 kg
Num rows    : 3100
Num Features: 8

 Cross-validation ---
CV R² mean: -0.1781
CV R² std : 0.4927
CV scores : [-0.062  0.4    0.092 -0.255 -1.065]

Grouped Cross-validation ---
Grouped CV R² mean: -0.0278
Grouped CV R² std : 0.5216
Grouped CV scores : [-0.053 -1.567  0.147 -0.013  0.227  0.231  0.2    0.166  0.15   0.233]
 EVALUATION: EXPERIMENT-3. SENTINEL + TANDEMX,Model: XGBOOST, Grouping? Yes, Hypertuned? Yes, Features: ['NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge']

Test set:
  R²   : 0.131
 

C:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best params: {'subsample': 1.0, 'reg_lambda': 5, 'reg_alpha': 1.0, 'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 10, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.5}

 LIGHTGBM: EXPERIMENT-1. SENTINEL + TANDEMX,Model: LightGBM, Grouping? Yes, Hypertuned? Yes, Features: ['height', 'species']
Test R²     : 0.2081
Test RMSE   : 4.11 kg
Train R² (log scale): 0.3792
Train R² (orig scale): 0.0452
Train RMSE  : 18.14 kg
Num rows    : 3100
Num Features: 4

 Cross-validation ---
CV R² mean: 0.2011
CV R² std : 0.2136
CV scores : [-0.089  0.571  0.2    0.114  0.21 ]

Grouped Cross-validation ---
Grouped CV R² mean: 0.3027
Grouped CV R² std : 0.1774
Grouped CV scores : [ 0.115  0.263  0.26  -0.028  0.349  0.537  0.629  0.296  0.265  0.341]
 EVALUATION: EXPERIMENT-1. SENTINEL + TANDEMX,Model: LightGBM, Grouping? Yes, Hypertuned? Yes, Features: ['height', 'species']

Test set:
  R²   : 0.208
  RMSE : 4.11 kg

 ✅ Test R² is positive (0.208)

Regular CrossValidation:
  Mea

# FIND THE BEST EXPERIMENT SO FAR.

## SUMMARY OF ALL EXPERIMENTS

In [65]:
tab_df = tabulate_results(global_experiment_list, top_n=None)

,model,Features,Groups?,Tuned?,Test R²,Test RMSE,Train R²,Train RMSE,CV R² Mean,CV R² Std,CV scores,Group CV R² Mean,Group CV R² Std,Group CV-Test Gap,-ve Folds,Group CV scores,verdict
0,MERF,"['EVI', 'NIR', 'NBR', 'SWIR1', 'height', 'species']",Yes,No,-4.015,10.34,0.1874,16.74,0.3191,0.1629,"[0.132, 0.454, 0.317, 0.149, 0.543]",0.333,0.172,4.348,1 (Big Creek),"[0.137, 0.334, 0.378, -0.058, 0.511, 0.538, 0.503, 0.32, 0.326, 0.341]",❌ REJECTED
1,MERF,"['EVI', 'NIR', 'NBR', 'SWIR1', 'height', 'species']",Yes,Yes,-3.611,9.91,0.1868,16.75,0.3138,0.1703,"[0.105, 0.457, 0.316, 0.148, 0.544]",0.328,0.1723,3.939,1 (Big Creek),"[0.108, 0.337, 0.376, -0.054, 0.504, 0.522, 0.505, 0.318, 0.327, 0.336]",❌ REJECTED
2,RANDOM FOREST,"['height', 'species']",Yes,Yes,-0.2016,5.06,0.1301,17.32,0.1409,0.2125,"[-0.215, 0.443, 0.223, 0.103, 0.151]",0.3092,0.1744,0.5107,1 (Big Creek_3),"[0.137, 0.243, 0.314, -0.1, 0.448, 0.496, 0.517, 0.363, 0.319, 0.355]",❌ REJECTED
3,MERF,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'height', 'species']",Yes,Yes,-3.375,9.65,0.1868,16.75,0.325,0.1775,"[0.103, 0.457, 0.348, 0.149, 0.568]",0.3079,0.1507,3.683,1 (Big Creek),"[0.082, 0.32, 0.354, -0.008, 0.478, 0.363, 0.507, 0.337, 0.348, 0.298]",❌ REJECTED
4,MERF,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'height', 'species']",Yes,No,-3.274,9.54,0.1874,16.74,0.33,0.1715,"[0.131, 0.459, 0.349, 0.144, 0.567]",0.3069,0.1492,3.581,1 (Big Creek),"[0.087, 0.326, 0.345, -0.009, 0.479, 0.353, 0.503, 0.335, 0.35, 0.299]",❌ REJECTED
5,LightGBM,"['height', 'species']",Yes,Yes,0.2081,4.11,0.0452,18.14,0.2011,0.2136,"[-0.089, 0.571, 0.2, 0.114, 0.21]",0.3027,0.1774,0.0946,1 (Big Creek_3),"[0.115, 0.263, 0.26, -0.028, 0.349, 0.537, 0.629, 0.296, 0.265, 0.341]",⚠️ MARGINAL — positive metrics but concerns
6,RANDOM FOREST,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,No,0.1997,4.13,0.1863,16.75,0.0555,0.3236,"[-0.349, 0.619, 0.153, -0.088, -0.057]",0.2996,0.2134,0.0999,"2 (Big Creek_5, Big Creek_3)","[-0.038, 0.105, 0.361, -0.017, 0.473, 0.571, 0.6, 0.355, 0.329, 0.258]",⚠️ MARGINAL — positive metrics but concerns
7,XGBOOST,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'height', 'species']",Yes,Yes,-0.0474,4.72,0.1639,16.98,0.1062,0.1771,"[0.06, 0.397, 0.157, -0.151, 0.068]",0.2951,0.1684,0.3425,1 (Big Creek_3),"[0.032, 0.19, 0.348, -0.019, 0.484, 0.48, 0.464, 0.339, 0.328, 0.305]",❌ REJECTED
8,RANDOM FOREST,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,Yes,0.1581,4.23,0.1718,16.9,0.1594,0.277,"[-0.182, 0.652, 0.215, 0.055, 0.056]",0.294,0.2022,0.1359,"2 (Big Creek_5, Big Creek_3)","[-0.019, 0.185, 0.298, -0.057, 0.465, 0.548, 0.571, 0.316, 0.272, 0.36]",⚠️ MARGINAL — positive metrics but concerns
9,XGBOOST,"['height', 'species']",Yes,Yes,0.0553,4.49,0.131,17.31,0.1922,0.0939,"[0.041, 0.319, 0.245, 0.147, 0.208]",0.2918,0.1533,0.2366,1 (Big Creek_3),"[0.163, 0.247, 0.297, -0.058, 0.441, 0.535, 0.366, 0.326, 0.246, 0.356]",⚠️ MARGINAL — positive metrics but concerns


## SUMMARY OF BEST ACCEPTABLE RESULTS

In [68]:
%run Model_functions.ipynb
best_results = filter_best_experiments(global_experiment_list)
tab_df = tabulate_results(best_results, top_n=10)

Found 132 experiments — best first.


,model,Features,Groups?,Tuned?,Test R²,Test RMSE,Train R²,Train RMSE,CV R² Mean,CV R² Std,CV scores,Group CV R² Mean,Group CV R² Std,Group CV-Test Gap,-ve Folds,Group CV scores,verdict
0,LightGBM,"['height', 'species']",Yes,Yes,0.2081,4.11,0.0452,18.14,0.2011,0.2136,"[-0.089, 0.571, 0.2, 0.114, 0.21]",0.3027,0.1774,0.0946,1 (Big Creek_3),"[0.115, 0.263, 0.26, -0.028, 0.349, 0.537, 0.629, 0.296, 0.265, 0.341]",⚠️ MARGINAL — positive metrics but concerns
1,RANDOM FOREST,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,No,0.1997,4.13,0.1863,16.75,0.0555,0.3236,"[-0.349, 0.619, 0.153, -0.088, -0.057]",0.2996,0.2134,0.0999,"2 (Big Creek_5, Big Creek_3)","[-0.038, 0.105, 0.361, -0.017, 0.473, 0.571, 0.6, 0.355, 0.329, 0.258]",⚠️ MARGINAL — positive metrics but concerns
2,RIDGE REGRESSION,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,Yes,0.3205,3.8,0.0403,18.19,-0.9117,2.177,"[-0.034, 0.679, 0.263, -0.244, -5.222]",0.2879,0.2124,0.0325,1 (Big Creek_3),"[0.142, 0.243, 0.134, -0.151, 0.455, 0.557, 0.621, 0.288, 0.289, 0.301]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
3,RIDGE REGRESSION,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,No,0.3205,3.8,0.0403,18.19,-0.9117,2.177,"[-0.034, 0.679, 0.263, -0.244, -5.222]",0.2879,0.2124,0.0325,1 (Big Creek_3),"[0.142, 0.243, 0.134, -0.151, 0.455, 0.557, 0.621, 0.288, 0.289, 0.301]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
4,RANDOM FOREST,"['height', 'species']",Yes,Yes,0.2993,3.86,0.0945,17.67,-0.9274,1.08,"[-0.396, -0.355, -0.346, -0.455, -3.086]",0.279,0.1674,0.0203,1 (Big Creek_3),"[0.236, 0.193, 0.088, -0.039, 0.472, 0.546, 0.43, 0.311, 0.255, 0.297]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
5,LASSO REGRESSION,"['EVI', 'NIR', 'NBR', 'SWIR1', 'height', 'species']",Yes,Yes,0.2264,4.06,0.0577,18.03,0.2099,0.2179,"[-0.032, 0.618, 0.113, 0.158, 0.193]",0.2785,0.1877,0.0521,none,"[0.05, 0.229, 0.103, 0.031, 0.359, 0.56, 0.616, 0.234, 0.263, 0.338]",⚠️ MARGINAL — positive metrics but concerns
6,LASSO REGRESSION,"['EVI', 'NIR', 'NBR', 'SWIR1', 'height', 'species']",Yes,No,0.2264,4.06,0.0577,18.03,0.2099,0.2179,"[-0.032, 0.618, 0.113, 0.158, 0.193]",0.2785,0.1877,0.0521,none,"[0.05, 0.229, 0.103, 0.031, 0.359, 0.56, 0.616, 0.234, 0.263, 0.338]",⚠️ MARGINAL — positive metrics but concerns
7,RANDOM FOREST,"['height', 'species']",Yes,No,0.2955,3.87,0.0974,17.64,-0.9727,1.062,"[-0.404, -0.361, -0.551, -0.455, -3.092]",0.2767,0.1649,0.0187,1 (Big Creek_3),"[0.238, 0.193, 0.089, -0.04, 0.472, 0.535, 0.419, 0.31, 0.254, 0.297]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
8,XGBOOST,"['height', 'species']",Yes,No,0.3177,3.81,0.0958,17.66,-0.3099,0.2245,"[-0.415, -0.588, 0.002, -0.454, -0.095]",0.2753,0.1655,0.0424,1 (Big Creek_3),"[0.237, 0.192, 0.089, -0.043, 0.469, 0.536, 0.422, 0.308, 0.248, 0.296]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
9,RANDOM FOREST,"['EVI', 'NIR', 'NBR', 'SWIR1', 'height', 'species']",Yes,No,0.2389,4.03,0.186,16.75,-0.0181,0.3516,"[-0.428, 0.527, 0.249, -0.231, -0.207]",0.2743,0.211,0.0355,"2 (Big Creek_5, Big Creek_3)","[-0.161, 0.199, 0.3

## BEST ONE

In [34]:
%run Model_functions.ipynb
best_result = best_results[0][1]
print_experiment(best_result)


 EXPERIMENT
  Model                : MERF
  Features             : ['height', 'species']
  Num Features         : 4
  Grouping             : Yes
  Stratified split     : No
  Hyperparameter tuned : Yes
  Test R²              : 0.3166
  Test RMSE            : 3.82 kg
  Train R² (log scale) : nan
  Train R² (orig scale): 0.1618
  Train RMSE           : 17.00 kg

 Cross-validation ---
  CV R² mean: 0.2811
  CV R² std : 0.1008
  CV scores : [0.234 0.388 0.273 0.122 0.389]

 Grouped Cross-validation ---
  Grouped CV R² mean : 0.2699
  Grouped CV R² std  : 0.1699
  Grouped CV scores  : [ 0.245  0.171  0.037 -0.012  0.449  0.558  0.398  0.342  0.194  0.318]

  Fold plots       : [['Big Creek_5', 'Channel Caye_1', 'Channel Caye_4', 'New River_2', 'Turneffe Atoll_3'], ['Frenchman Caye_6', 'Gra Gra Lagoon_6', 'Hicks Caye_2', 'Hicks Caye_6', 'Turneffe Atoll_4'], ['Frenchman Caye_3', 'Gra Gra Lagoon_3', 'Hicks Caye_4', "Payne's Creek_3", 'Shipstern Lagoon_6'], ['Big Creek_3', 'Channel Caye_3', 'C

# SAVE EXPERIMENT RESULTS

In [73]:
import pickle

with open('sentinel_experiments.pkl', 'wb') as f:
    pickle.dump(global_experiment_list, f)

In [240]:
loaded_dict = {}
with open('sentinel_experiments.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)

## Verify loaded values

In [242]:
%run Model_functions.ipynb
verify_dicts(global_experiment_list, loaded_dict)

All values match!


## Check if the loaded values are printabled

### Print the first experiment.

In [287]:
%run Model_functions.ipynb
key_list = list(loaded_dict.keys())
print_experiment(loaded_dict, key_list[0])


DATASET: SENTINEL + TANDEMX
  Model                : LINEAR REGRESSION
  Features             : ['height', 'species']
  Grouping             : Yes
  Stratified split     : No
  Hyperparameter tuned : No
  Test R²              : 0.0724
  Test RMSE            : 4.45 kg
  Train R² (log scale) : nan
  Train R² (orig scale): -0.0202
  Train RMSE           : 18.76 kg

 Cross-validation ---
  CV R² mean: 0.2144
  CV R² std : 0.2030
  CV scores : [-0.023  0.59   0.182  0.133  0.189]

 Grouped Cross-validation ---
  Grouped CV R² mean : 0.2626
  Grouped CV R² std  : 0.2058
  Grouped CV scores  : [-0.046  0.186  0.123 -0.01   0.394  0.54   0.621  0.221  0.268  0.328]

  Fold plots       : [['Big Creek_5', 'Channel Caye_1', 'Channel Caye_4', 'New River_2', 'Turneffe Atoll_3'], ['Frenchman Caye_6', 'Gra Gra Lagoon_6', 'Hicks Caye_2', 'Hicks Caye_6', 'Turneffe Atoll_4'], ['Frenchman Caye_3', 'Gra Gra Lagoon_3', 'Hicks Caye_4', "Payne's Creek_3", 'Shipstern Lagoon_6'], ['Big Creek_3', 'Channel Caye

### Display the best experiments from the loaded results

In [303]:
%run Model_functions.ipynb
best_results = filter_best_experiments(loaded_dict)
tab_df = tabulate_results(best_results, top_n=10)

Found 132 experiments — best first.


,Dataset,Model,Features,Groups?,Tuned?,Test R²,Test RMSE,Train R²,Train RMSE,CV R² Mean,CV R² Std,CV scores,Group CV R² Mean,Group CV R² Std,Group CV-Test Gap,-ve Folds,Group CV scores,verdict
0,{'SENTINEL + TANDEMX'},LightGBM,"['height', 'species']",Yes,Yes,0.2081,4.11,0.0452,18.14,0.2011,0.2136,"[-0.089, 0.571, 0.2, 0.114, 0.21]",0.3027,0.1774,0.0946,1 (Big Creek_3),"[0.115, 0.263, 0.26, -0.028, 0.349, 0.537, 0.629, 0.296, 0.265, 0.341]",⚠️ MARGINAL — positive metrics but concerns
1,{'SENTINEL + TANDEMX'},RANDOM FOREST,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,No,0.1997,4.13,0.1863,16.75,0.0555,0.3236,"[-0.349, 0.619, 0.153, -0.088, -0.057]",0.2996,0.2134,0.0999,"2 (Big Creek_5, Big Creek_3)","[-0.038, 0.105, 0.361, -0.017, 0.473, 0.571, 0.6, 0.355, 0.329, 0.258]",⚠️ MARGINAL — positive metrics but concerns
2,{'SENTINEL + SIMARD'},RIDGE REGRESSION,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,Yes,0.3205,3.8,0.0403,18.19,-0.9117,2.177,"[-0.034, 0.679, 0.263, -0.244, -5.222]",0.2879,0.2124,0.0325,1 (Big Creek_3),"[0.142, 0.243, 0.134, -0.151, 0.455, 0.557, 0.621, 0.288, 0.289, 0.301]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
3,{'SENTINEL + SIMARD'},RIDGE REGRESSION,"['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'NIR', 'SWIR1', 'SWIR2', 'NDVI', 'MNDWI', 'NBR', 'EVI', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge', 'height', 'height_x_EVI', 'height_x_NBR', 'height_x_NDVI', 'height_x_NDRE1', 'height_x_NDRE2', 'height_x_NDRE3', 'height_x_CIrededge', 'height_x_NIR', 'height_x_SWIR1', 'height_x_SWIR2', 'species']",Yes,No,0.3205,3.8,0.0403,18.19,-0.9117,2.177,"[-0.034, 0.679, 0.263, -0.244, -5.222]",0.2879,0.2124,0.0325,1 (Big Creek_3),"[0.142, 0.243, 0.134, -0.151, 0.455, 0.557, 0.621, 0.288, 0.289, 0.301]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
4,{'SENTINEL + SIMARD'},RANDOM FOREST,"['height', 'species']",Yes,Yes,0.2993,3.86,0.0945,17.67,-0.9274,1.08,"[-0.396, -0.355, -0.346, -0.455, -3.086]",0.279,0.1674,0.0203,1 (Big Creek_3),"[0.236, 0.193, 0.088, -0.039, 0.472, 0.546, 0.43, 0.311, 0.255, 0.297]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
5,{'SENTINEL + TANDEMX'},LASSO REGRESSION,"['EVI', 'NIR', 'NBR', 'SWIR1', 'height', 'species']",Yes,Yes,0.2264,4.06,0.0577,18.03,0.2099,0.2179,"[-0.032, 0.618, 0.113, 0.158, 0.193]",0.2785,0.1877,0.0521,none,"[0.05, 0.229, 0.103, 0.031, 0.359, 0.56, 0.616, 0.234, 0.263, 0.338]",⚠️ MARGINAL — positive metrics but concerns
6,{'SENTINEL + TANDEMX'},LASSO REGRESSION,"['EVI', 'NIR', 'NBR', 'SWIR1', 'height', 'species']",Yes,No,0.2264,4.06,0.0577,18.03,0.2099,0.2179,"[-0.032, 0.618, 0.113, 0.158, 0.193]",0.2785,0.1877,0.0521,none,"[0.05, 0.229, 0.103, 0.031, 0.359, 0.56, 0.616, 0.234, 0.263, 0.338]",⚠️ MARGINAL — positive metrics but concerns
7,{'SENTINEL + SIMARD'},RANDOM FOREST,"['height', 'species']",Yes,No,0.2955,3.87,0.0974,17.64,-0.9727,1.062,"[-0.404, -0.361, -0.551, -0.455, -3.092]",0.2767,0.1649,0.0187,1 (Big Creek_3),"[0.238, 0.193, 0.089, -0.04, 0.472, 0.535, 0.419, 0.31, 0.254, 0.297]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
8,{'SENTINEL + SIMARD'},XGBOOST,"['height', 'species']",Yes,No,0.3177,3.81,0.0958,17.66,-0.3099,0.2245,"[-0.415, -0.588, 0.002, -0.454, -0.095]",0.2753,0.1655,0.0424,1 (Big Creek_3),"[0.237, 0.192, 0.089, -0.043, 0.469, 0.536, 0.422, 0.308, 0.248, 0.296]",⚠️ MARGINAL (grouped CV clean but regular CV negative)
9,{'SENTINEL + TAN

### Display experiment from a chosen row among the best list

In [306]:
%run Model_functions.ipynb
print_experiment_from_row(tab_df.iloc[0])


Dataset             : {'SENTINEL + TANDEMX'}
  Model               : LightGBM
  Features            : ['height', 'species']
  Grouping            : Yes
  Stratified split    : None
  Hyperparameter tuned: Yes
  Test R²     : 0.2081
  Test RMSE   : 4.11 kg
  Train R²    : 0.0452
  Train RMSE  : 18.14 kg

 Cross-validation ---
  CV R² mean: 0.2011
  CV R² std : 0.2136
  CV scores : [-0.089  0.571  0.2    0.114  0.21 ]

Grouped Cross-validation ---
  Grouped CV R² mean : 0.3027
  Grouped CV R² std  : 0.1774
  Grouped CV scores  : [ 0.115  0.263  0.26  -0.028  0.349  0.537  0.629  0.296  0.265  0.341]
  Grouped CV-Test Gap: 0.0946
  -ve folds          : 1 (Big Creek_3)

  Verdict: ⚠️  MARGINAL — positive metrics but concerns
